# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [9]:
# imports
import os
import json
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
#from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [10]:
# constants

MODEL_GEMINI = 'gemini-2.5-flash'
MODEL_LLAMA = 'llama3.2'
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [19]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv("GEMINI_API_KEY")
requests.get("http://localhost:11434").content

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("AQ."):
    print("An API key was found, but it doesn't start AQ.; please check you're using the right key")
else:
    print("GEMINI API key found and looks good so far!")

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="Ollama")

system_prompt = """
You are a helpful senior developer with plenty knownlegde in coding. 
your job is to analyze and explain coding related question to junior developers.
Respond in markdown without code blocks.
Include details of code language and libraries that might have been used or any other
necessary information to understan hundred percent the code problem.
"""


def ask_question_to_ollama(question):
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": question}]
    
    response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages)
    markdown_text = response.choices[0].message.content
    return display(Markdown(markdown_text))

def ask_question_to_gemini(question):
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": question}]

    stream = gemini.chat.completions.create(model=MODEL_GEMINI, messages=messages, stream=True)
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

    

GEMINI API key found and looks good so far!


In [12]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [13]:
# Get gpt-4o-mini to answer, with streaming
ask_question_to_gemini(question)

This code snippet is written in **Python**, and it's typically found within a generator function. It elegantly extracts unique author names from a collection of book data and then yields each of those names.

Let's break down what it does and why:

### 1. The Inner Part: Set Comprehension

First, let's look at the part inside the `yield from`:
`{book.get("author") for book in books if book.get("author")}`

This is a **set comprehension**. It's a concise way to create a Python `set` based on an iterable.

*   **`for book in books`**: This iterates through an iterable called `books`. We can infer that `books` is likely a list, tuple, or another collection where each element (`book`) is expected to be a dictionary or an object that behaves like one (i.e., it has a `.get()` method).

*   **`book.get("author")`**: For each `book`, it attempts to retrieve the value associated with the key `"author"`. The `.get()` method is used here instead of direct access (`book["author"]`) because it's safer. If a `book` dictionary doesn't have an "author" key, `book.get("author")` will return `None` instead of raising a `KeyError`.

*   **`if book.get("author")`**: This is a filter condition. It checks if the value returned by `book.get("author")` is "truthy". In Python, `None`, empty strings (`""`), `False`, and `0` are considered "falsy". So, this condition effectively:
    *   Excludes books that don't have an "author" key (because `get("author")` would return `None`).
    *   Excludes books where the author value is empty or otherwise falsy (e.g., if an author was explicitly stored as `""` or `None`).
    *   Only processes books where an actual, non-empty author name is present.

*   **`{ ... }`**: The curly braces signify that the result of this comprehension is a **set**. A key property of sets is that they only store **unique** elements. If multiple books share the same author, that author's name will only appear once in the resulting set.

**In summary of the inner part**: This set comprehension iterates through your `books` collection, extracts the author name for each book (if an author is present and truthy), and collects all these unique author names into a Python set.

### 2. The Outer Part: `yield from`

Now, let's consider the `yield from` statement:
`yield from { ... }`

*   **`yield from`**: This is a Python feature used within generator functions. It's designed to delegate iteration to a sub-iterable (or sub-generator). Instead of manually looping and yielding each item, `yield from` handles it concisely.

    If you had a set named `unique_authors_set`, writing `yield from unique_authors_set` is equivalent to:
    ```python
    for author_name in unique_authors_set:
        yield author_name
    ```

**In summary of the outer part**: The `yield from` keyword takes the set of unique author names (generated by the set comprehension) and *yields each author name one by one* as part of the sequence generated by the containing generator function.

### What it does overall and why:

This line of code, found inside a Python generator function, performs the following tasks:

1.  **Filters**: It processes a collection of `books`, ignoring any `book` that does not have a valid, non-empty author entry.
2.  **Extracts**: For the valid books, it extracts their author names.
3.  **Deduplicates**: By using a set comprehension, it automatically ensures that only unique author names are collected. If "John Doe" authored five books in your `books` collection, "John Doe" will only appear once in the resulting set.
4.  **Yields Iteratively**: Finally, `yield from` takes this set of unique authors and turns it into a stream of individual author names. The generator function will yield each unique author name sequentially to its caller.

**Why is this useful?**

*   **Efficiency for Large Datasets**: If `books` is a very large collection, `yield from` combined with a generator function is memory-efficient. You're not building an entire list of all authors (potentially with duplicates) in memory before processing. While the set *is* built in memory, it only holds unique authors, which could be significantly smaller than the total number of books if there are many duplicate authors. The `yield from` then processes this set item by item.
*   **Conciseness**: It's a very compact and Pythonic way to achieve filtering, deduplication, and iterative yielding of elements.
*   **Readability**: Once you understand set comprehensions and `yield from`, the code clearly communicates its intent: "give me the unique, valid authors from these books, one at a time."
*   **Data Cleaning**: The `if book.get("author")` clause acts as a quick data cleaner, ensuring you only work with meaningful author data.

This pattern is very common in data processing tasks where you need to extract unique attributes from a collection of records in an efficient and clean manner.

In [20]:
# Get Llama 3.2 to answer
ask_question_to_ollama(question)

**Explanation**

This code snippet utilizes a combination of Python's built-in functions and dictionaries, as well as the `yield from` keyword.

The outer expression `{...}` is used to create a dictionary comprehension. It takes an iterable (`books`) as input and returns another iterable, where each item in the original list is processed according to the provided function.

Inside the dictionary comprehension, the `.get("author")` method is called on each book object in the `books` list. This returns a value associated with the key `"author"` in the book's dictionary-like attribute (i.e., `book`).

The outer expression `yield from {...}` introduces a new concept: the yield from keyword.

In Python 3.3 and later, the `yield from` statement allows you to delegate a sub-iteration using the yielded value to an iterabled object within a generator function. In essence, it changes a generator back into an iterator for that child iteration (instead of just yielding its objects).

**How it works**

When `yield from {book.get("author") for book in books if book.get("author")}` is executed:

1. The dictionary comprehension `{...}` processes each book in `books` and returns an iterable containing the author name for each valid book.
2. The outer expression's result becomes the input (i.e., a new iterator) to `yield from`.
3. When iterating over this resulting object, Python will call `.get("author")` on the current book object.
4. If the iteration yields an author's name, it is yielded by Python as well.

**Practical usage**

This code snippet can be used in several scenarios:

*   When you have a large list of books with incomplete or inconsistent data and need to extract a subset of only the books that contain `"author"`.
*   When processing large datasets and want to iterate over only valid or meaningful records, while keeping the structure clear.

Here's an example for better understanding:
```python
# Mock book class
class Book:
    def __init__(self, title, author):
        self.title = title
        self.author = author

books = [
    {"author": "Zorvath"},
    {  # incomplete book: no 'author'
        # some attribute present (title)
        },
    {"author": None},
]

# Dictionary comprehension as function
def filter_books_by_author(books):
    """Returns an iterable of authors for complete books"""
    return {book.get("author") for book in books if book.get("author") != ""}

# Using yield from with dictionary comprehension
for author in yield_from ({book.get("author") for book in books if book.get("author")}):
    print(author)
```